In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def dcg_at_k(rels, k):
    rels = np.asfarray(rels)[:k]
    if rels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels.size + 2))
    return np.sum(rels / discounts)

def ndcg_at_k(binary_rels, k):
    # binary_rels: list/array of 0/1 relevance in ranked order
    dcg = dcg_at_k(binary_rels, k)
    ideal = dcg_at_k(sorted(binary_rels, reverse=True), k)
    return 0.0 if ideal == 0 else float(dcg / ideal)

def recall_at_k(recommended, relevant_set, k):
    if not relevant_set:
        return np.nan
    rec_k = recommended[:k]
    hits = len(set(rec_k) & relevant_set)
    return hits / len(relevant_set)

def precision_at_k(recommended, relevant_set, k):
    rec_k = recommended[:k]
    if k == 0:
        return np.nan
    hits = len(set(rec_k) & relevant_set)
    return hits / k

def average_precision_at_k(recommended, relevant_set, k):
    if not relevant_set:
        return np.nan
    rec_k = recommended[:k]
    hits = 0
    sum_prec = 0.0
    for i, item in enumerate(rec_k, start=1):
        if item in relevant_set:
            hits += 1
            sum_prec += hits / i
    return sum_prec / min(len(relevant_set), k)

In [ ]:
# === EDIT THESE 3 IF NEEDED ===
RATINGS_DF_NAME = "ratings_df"   # <-- ganti kalau nama df beda
USER_COL = "user_id"
ITEM_COL = "game_id"
RATING_COL = "rating"

ratings_df = globals().get(RATINGS_DF_NAME)
assert ratings_df is not None, f"DataFrame '{RATINGS_DF_NAME}' tidak ketemu di notebook."

# threshold relevance (umum: rating >= 4 dianggap relevant)
RELEVANCE_THRESHOLD = 4.0

# user -> set of relevant items
relevant_by_user = (
    ratings_df[ratings_df[RATING_COL] >= RELEVANCE_THRESHOLD]
    .groupby(USER_COL)[ITEM_COL]
    .apply(lambda s: set(s.tolist()))
    .to_dict()
)

# Also: items already seen by user (for filtering)
seen_by_user = (
    ratings_df.groupby(USER_COL)[ITEM_COL]
    .apply(lambda s: set(s.tolist()))
    .to_dict()
)

len(relevant_by_user), len(seen_by_user)

In [ ]:
# Users that have at least 1 relevant item
eval_users = [u for u, rel in relevant_by_user.items() if len(rel) > 0]

print("Total eval users:", len(eval_users))
print("Example user:", eval_users[0] if eval_users else None)

In [ ]:
# === You need an item universe ===
# ganti ITEMS_DF_NAME / ITEM_ID_COL sesuai dataset kamu (games metadata df)
ITEMS_DF_NAME = "games_df"    # <-- ganti kalau beda
ITEM_ID_COL = "game_id"

games_df = globals().get(ITEMS_DF_NAME)
assert games_df is not None, f"DataFrame '{ITEMS_DF_NAME}' tidak ketemu. Set item universe dulu."
all_items = games_df[ITEM_ID_COL].dropna().unique().tolist()

rng = np.random.default_rng(42)

def recommend(user_id, k=10):
    seen = seen_by_user.get(user_id, set())
    candidates = [i for i in all_items if i not in seen]
    if len(candidates) == 0:
        return []
    # random baseline
    recs = rng.choice(candidates, size=min(k, len(candidates)), replace=False).tolist()
    return recs

In [ ]:
KS = [5, 10, 20]
results = []

for u in eval_users:
    recs_50 = recommend(u, k=50)  # compute once, slice for K
    rel = relevant_by_user[u]

    # build binary relevance list aligned with rec order (for ndcg)
    binary_rels_50 = [1 if it in rel else 0 for it in recs_50]

    row = {"user": u, "n_relevant": len(rel), "n_seen": len(seen_by_user.get(u, set()))}
    for k in KS:
        row[f"Recall@{k}"] = recall_at_k(recs_50, rel, k)
        row[f"Precision@{k}"] = precision_at_k(recs_50, rel, k)
        row[f"MAP@{k}"] = average_precision_at_k(recs_50, rel, k)
        row[f"NDCG@{k}"] = ndcg_at_k(binary_rels_50[:k], k)
    results.append(row)

eval_df = pd.DataFrame(results)

summary = eval_df[[c for c in eval_df.columns if c.startswith(("Recall@", "Precision@", "MAP@", "NDCG@"))]].mean(numeric_only=True)
display(eval_df.head())
summary

In [ ]:
USER_COLD_N = 5
ITEM_COLD_N = 10

# user interaction counts
user_counts = ratings_df.groupby(USER_COL)[ITEM_COL].count().to_dict()
eval_df["is_user_cold"] = eval_df["user"].map(lambda u: user_counts.get(u, 0) < USER_COLD_N)

# item rating counts (optional)
item_counts = ratings_df.groupby(ITEM_COL)[USER_COL].count().to_dict()

def user_relevant_cold_ratio(u):
    rel = relevant_by_user.get(u, set())
    if not rel:
        return np.nan
    cold_rel = sum(1 for it in rel if item_counts.get(it, 0) < ITEM_COLD_N)
    return cold_rel / len(rel)

eval_df["relevant_item_cold_ratio"] = eval_df["user"].map(user_relevant_cold_ratio)

# Compare metrics: cold vs non-cold
metric_cols = [c for c in eval_df.columns if c.startswith(("Recall@", "Precision@", "MAP@", "NDCG@"))]
cold_summary = eval_df.groupby("is_user_cold")[metric_cols].mean(numeric_only=True)

cold_summary